# 2hClub — ball detector training (Colab)

Trains the ball detector. Round 1: `ball_tiles.zip` from `vision/build_dataset.py`.
Round 2: `ball_tiles_r2.zip` from `vision/build_dataset_r2.py` (round 1 + hard examples).

1. Runtime → Change runtime type → **GPU** (L4 or A100 if you have them; T4 works, just slower).
2. Upload the zip anywhere in your Google Drive (the notebook finds it).
3. Run all cells. Progress is saved to `MyDrive/2hclub/runs/<RUN_NAME>/` after every epoch, so a disconnect loses nothing.

The tiles are clipped match video: keep the zip and the Drive folder **private** (RULES.md, terms of data use).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Settings. Ball is ~6 px in a 640 tile, so we train at 1280 (the ball becomes ~12 px).
DRIVE_DIR = '/content/drive/MyDrive/2hclub'   # results go here
ZIP_NAME = 'ball_tiles_r2.zip'   # round 1 was 'ball_tiles.zip'
MODEL = 'yolo26s.pt'      # small; try yolo26m.pt if there is time and GPU memory
IMGSZ = 1280
EPOCHS = 25               # round 1 used 25
BATCH = 16                # fits an L4; the trainer drops it to 8 by itself on a T4
RUN_NAME = 'ball_r2'
SEED = 0
RUNS_DIR = '/content/drive/MyDrive/2hclub/runs'   # straight to Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.190

In [ ]:
import os, shutil, zipfile
os.makedirs('/content/yolo', exist_ok=True)
import glob
# Find the zip wherever it was uploaded in Drive.
found = glob.glob(f'/content/drive/MyDrive/**/{ZIP_NAME}', recursive=True)
assert found, f'{ZIP_NAME} not found anywhere in your Google Drive: upload it first'
print('using', found[0])
with zipfile.ZipFile(found[0]) as z:
    z.extractall('/content/yolo')
# data.yaml from the Mac points at a local path; rewrite it for Colab.
with open('/content/yolo/data.yaml', 'w') as f:
    f.write('path: /content/yolo\ntrain: train/images\nval: val/images\nnames:\n  0: ball\n')
for split in ('train', 'val'):
    imgs = os.listdir(f'/content/yolo/{split}/images')
    print(split, len(imgs), 'tiles,', sum('_neg' in i or '_fp' in i for i in imgs), 'ball-free')

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data='/content/yolo/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    seed=SEED, deterministic=True, patience=15, cos_lr=True,
    # Tiny objects: no heavy zoom-out (it would shrink the ball to 2-3 px), keep flips and colour.
    scale=0.3, mosaic=1.0, close_mosaic=10, fliplr=0.5, degrees=0.0,
    project=RUNS_DIR, name=RUN_NAME, exist_ok=True, plots=True, save_period=1,
)

In [ ]:
best = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
metrics = YOLO(best).val(data='/content/yolo/data.yaml', imgsz=IMGSZ, batch=BATCH, conf=0.001)
print('precision', metrics.box.mp, 'recall', metrics.box.mr, 'mAP50', metrics.box.map50)
print('weights in', best)

In [ ]:
# Eyeball a few validation tiles: green = prediction.
import random, cv2, matplotlib.pyplot as plt
model = YOLO(best)
files = random.Random(0).sample([f for f in os.listdir('/content/yolo/val/images') if '_neg' not in f and '_fp' not in f], 8)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, name in zip(axes.ravel(), files):
    r = model.predict(f'/content/yolo/val/images/{name}', imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    img = cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2RGB)
    for x1, y1, x2, y2 in r.boxes.xyxy.cpu().numpy().astype(int):
        cv2.rectangle(img, (x1 - 6, y1 - 6), (x2 + 6, y2 + 6), (0, 255, 0), 2)
    ax.imshow(img); ax.set_title(f'{name}  {len(r.boxes)} det'); ax.axis('off')
plt.tight_layout()